# AI Agent with Lakebase Memory — OpenAI Agents SDK + Unity Gateway

The LLM-powered version of the training demo: the same Lakebase Postgres session memory, but the agent's reply step is now a **real LLM call** made with the **OpenAI Agents SDK** and routed through **Databricks Unity Gateway** to a low-cost, governed model service (`system.ai.llama-4-maverick`). Runs on serverless compute, Python only.

## What changes vs. the first demo

In *Lakebase Agent Session Memory Demo*, the reply step was hardcoded Python — reliable, but not an agent in the LLM sense. This notebook swaps in the real pieces:

* **OpenAI Agents SDK** (`openai-agents`) — the open-source agent framework: define an `Agent` with instructions, run it with `Runner`. No lock-in to a proprietary API.
* **Unity Gateway** — the agent talks to the model through Databricks' OpenAI-compatible API (`<workspace>/ai-gateway/mlflow/v1`). Why it matters:
  * The model is a **governed Unity Catalog model service** (`system.ai.llama-4-maverick`) — open-weights and low-cost, ideal for a demo.
  * **No API keys** to manage — the workspace's own credentials authenticate the call.
  * **Usage is tracked** (`system.ai_gateway.usage`), so you can show the audience exactly what the demo cost.
* **The memory loop is unchanged** — retrieve history from Lakebase, pass it to the LLM as context, persist both sides of the conversation back to Lakebase. Lakebase is still the *only* thing the agent remembers between runs.

The punchline is the same: a brand-new agent instance (cold start, restart, redeploy) still knows who you are — because memory lives in Postgres, not in the process or the model.

In [0]:
%pip install openai-agents pg8000
# Pure-Python packages (openai + agents + pg8000 Postgres driver) — safe on
# serverless compute. NOTE: do NOT install psycopg/psycopg2 here; they crash
# the serverless kernel (SIGABRT), which is why we use pg8000 (as in demo 1).

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 90.5 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Not uninstalling urllib3 at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-b2c4d376-dafe-4be7-960d-317d7eb867fa
    Can't uninstall 'urllib3'. No files were found to uninstall.
  Attempting uninstall: pyjwt
    Found existing installation: PyJWT 2.10.1
    Not uninstalling pyjwt at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-b2c4d376-dafe-4be7-960d-317d7eb867fa
    Can't uninstall 'PyJWT'. No files were found to uninstall.
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Not uninstalling idna at /databricks/python3/lib/python3.12/site-packages, outside environment /loca

In [0]:
# Lakebase connection helper — IDENTICAL construct to demo 1.
# Fresh connection per operation, OAuth token minted from the Databricks
# identity each time, no passwords anywhere.
import ssl

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()  # pre-authenticated with your Databricks identity

LAKEBASE_HOST = "ep-cold-forest-d8ignwgw.database.us-east-2.cloud.databricks.com"
LAKEBASE_DATABASE = "databricks_postgres"
LAKEBASE_ENDPOINT = "projects/alan/branches/production/endpoints/primary"


def get_connection():
    """Open a fresh connection to the Lakebase database (pg8000 driver)."""
    import pg8000.native

    token = w.postgres.generate_database_credential(
        endpoint=LAKEBASE_ENDPOINT
    ).token  # NEVER print or log this token

    ctx = ssl.create_default_context()  # equivalent of sslmode=require
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE

    return pg8000.native.Connection(
        user=w.current_user.me().user_name,
        host=LAKEBASE_HOST,
        database=LAKEBASE_DATABASE,
        password=token,
        ssl_context=ctx,
        timeout=30,
    )


# Smoke test
conn = get_connection()
rows = conn.run("SELECT current_database(), current_user, version()")
conn.close()
print("Connected to:", rows[0][0], "| logged in as:", rows[0][1])

Connected to: databricks_postgres | logged in as: alan.dennis@databricks.com


In [0]:
# Unity Gateway setup — point the OpenAI SDK at Databricks.
#
# The Gateway exposes an OpenAI-compatible Chat Completions API at
#   <workspace-host>/ai-gateway/mlflow/v1
# so the OpenAI Agents SDK works unmodified — we just give it our own client.
from agents import set_tracing_disabled
from openai import AsyncOpenAI

MODEL = "system.ai.llama-4-maverick"  # low-cost, open-weights, governed UC model service

# Mint the workspace token from the SDK's auth headers.
# (Notebook runtime auth doesn't expose oauth_token(); the same bearer the
# SDK uses for every API call is in the authenticate() headers.)
# NEVER print or log the token.
auth_headers = w.config.authenticate()
_session_token = auth_headers["Authorization"].removeprefix("Bearer ")
print("Gateway token minted: True (valid ~1 hour — mint again if the demo runs long)")

gw_client = AsyncOpenAI(
    api_key=_session_token,
    base_url=f"{w.config.host.rstrip('/')}/ai-gateway/mlflow/v1",
)
set_tracing_disabled(True)  # tracing to the OpenAI platform is not available here
print("OpenAI-compatible client pointed at Unity Gateway.")

Gateway token minted: True (valid ~1 hour — mint again if the demo runs long)
OpenAI-compatible client pointed at Unity Gateway.


In [0]:
# Smoke test: one agent run through the Gateway before we build the demo.
from agents import Agent, ModelSettings, OpenAIChatCompletionsModel, Runner

probe_agent = Agent(
    name="ProbeAgent",
    instructions="You are a terse assistant. Reply with exactly what is asked.",
    model=OpenAIChatCompletionsModel(model=MODEL, openai_client=gw_client),
    model_settings=ModelSettings(max_tokens=32),
)

# The Agents SDK Runner is async — notebooks (IPython) support top-level await.
probe_result = await Runner.run(probe_agent, "Reply with exactly: OK")
print("Gateway + Agents SDK smoke test:", probe_result.final_output)

Gateway + Agents SDK smoke test: OK


In [0]:
# Memory schema — same design as demo 1, but a SEPARATE table so the two
# notebooks don't wipe each other's data when re-run. DROP first for re-runnability.
conn = get_connection()
conn.run("DROP TABLE IF EXISTS agent_session_memory_llm")
conn.run(
    """
    CREATE TABLE agent_session_memory_llm (
        session_id  TEXT        NOT NULL,
        turn        INTEGER     NOT NULL,
        role        TEXT        NOT NULL,   -- 'user' or 'agent'
        content     TEXT        NOT NULL,
        created_at  TIMESTAMPTZ DEFAULT now(),
        PRIMARY KEY (session_id, turn)
    )
    """
)
conn.close()
print("Table agent_session_memory_llm created.")

Table agent_session_memory_llm created.


In [0]:
# SessionMemory — IDENTICAL construct to demo 1 (only the table differs).
class SessionMemory:
    """Persistent conversation memory backed by a Lakebase Postgres table.

    Every method opens a fresh connection (minting a fresh OAuth token),
    so it works no matter which process — or which replica — calls it.
    """

    def __init__(self, table):
        self.table = table

    def save_turn(self, session_id, turn, role, content):
        """INSERT one turn of the conversation."""
        conn = get_connection()
        conn.run(
            f"INSERT INTO {self.table} (session_id, turn, role, content) "
            "VALUES (:session_id, :turn, :role, :content)",
            session_id=session_id,
            turn=turn,
            role=role,
            content=content,
        )
        conn.close()

    def get_history(self, session_id):
        """SELECT all turns for a session, ordered by turn."""
        conn = get_connection()
        rows = conn.run(
            f"SELECT turn, role, content FROM {self.table} "
            "WHERE session_id = :session_id ORDER BY turn",
            session_id=session_id,
        )
        conn.close()
        return [{"turn": t, "role": r, "content": c} for (t, r, c) in rows]

    def forget_session(self, session_id):
        """DELETE every turn for a session (a full memory wipe)."""
        conn = get_connection()
        conn.run(
            f"DELETE FROM {self.table} WHERE session_id = :session_id",
            session_id=session_id,
        )
        conn.close()


memory = SessionMemory(table="agent_session_memory_llm")
print("SessionMemory ready (table: agent_session_memory_llm).")

SessionMemory ready (table: agent_session_memory_llm).


In [0]:
# The agent — same three-step loop as demo 1, but the reply comes from an
# LLM through the OpenAI Agents SDK.
from agents import Agent, ModelSettings, OpenAIChatCompletionsModel, Runner


llm_agent = Agent(
    name="MemoryAgent",
    instructions=(
        "You are a friendly assistant in a live demo. You have persistent "
        "memory: the conversation so far is given to you as context. Keep "
        "replies to one or two sentences. When the user asks what their name "
        "is or what they like, answer from the conversation context."
    ),
    model=OpenAIChatCompletionsModel(model=MODEL, openai_client=gw_client),
    model_settings=ModelSettings(max_tokens=200),
)


class DemoAgent:
    """Agent with Lakebase-backed session memory and an LLM reply step.

    On every turn it:
      1. Retrieves the full prior history from Lakebase (its 'context'),
      2. Runs the OpenAI Agents SDK Agent on the retrieved history + the new
         message — this is where Lakebase memory becomes LLM context,
      3. Persists BOTH the user's message and the agent's reply to Lakebase.
    """

    def __init__(self, session_id, memory, agent):
        self.session_id = session_id
        self.memory = memory
        self.agent = agent

    async def reply(self, user_text):
        # 1) Retrieve prior turns from Postgres — this is the agent's memory.
        history = self.memory.get_history(self.session_id)

        # 2) LLM call with the retrieved history as context.
        if history:
            convo = "\n".join(f"{t['role']}: {t['content']}" for t in history)
            prompt = f"Conversation so far:\n{convo}\n\nuser: {user_text}"
        else:
            prompt = user_text
        result = await Runner.run(self.agent, prompt)
        response = result.final_output

        # 3) Persist both sides of the conversation to Postgres.
        next_turn = (history[-1]["turn"] + 1) if history else 1
        self.memory.save_turn(self.session_id, next_turn, "user", user_text)
        self.memory.save_turn(self.session_id, next_turn + 1, "agent", response)
        return response


print("DemoAgent (LLM-backed) ready.")

DemoAgent (LLM-backed) ready.


In [0]:
# --- Session 1: a first conversation ---
# Same flow as demo 1 — but now the replies come from the LLM, and every
# turn (user + agent) is saved to Lakebase.
SESSION_ID = "llm-demo-session-1"

agent = DemoAgent(SESSION_ID, memory, llm_agent)

for user_message in ["Hi, I'm Alan", "I like tea."]:
    print(f"User : {user_message}")
    print(f"Agent: {await agent.reply(user_message)}\n")

print("What the agent just stored in Postgres:")
for t in memory.get_history(SESSION_ID):
    print(f"  turn {t['turn']:>2} [{t['role']:>5}] {t['content']}")

User : Hi, I'm Alan
Agent: Nice to meet you, Alan!

User : I like tea.
Agent: That's great, Alan, there's a lovely tea shop just around the corner from here.

What the agent just stored in Postgres:
  turn  1 [ user] Hi, I'm Alan
  turn  2 [agent] Nice to meet you, Alan!
  turn  3 [ user] I like tea.
  turn  4 [agent] That's great, Alan, there's a lovely tea shop just around the corner from here.


In [0]:
# --- Simulate a total agent restart ---
# Destroy the in-process agent. The brand-new instance has ZERO in-memory
# state — and unlike the model, which is stateless between calls, only
# Lakebase holds the memory. Same punchline as demo 1, now with an LLM.
del agent

agent2 = DemoAgent(SESSION_ID, memory, llm_agent)  # same session, fresh "process"

user_message = "What's my name? What do I like?"
print(f"User : {user_message}")
print(f"Agent: {await agent2.reply(user_message)}")
print()
print("The new agent instance answered from Lakebase memory — the LLM alone would have forgotten.")

# Show the raw table: everything the agent has ever stored.
conn = get_connection()
rows = conn.run(
    "SELECT session_id, turn, role, content, created_at "
    "FROM agent_session_memory_llm ORDER BY session_id, turn"
)
conn.close()
print("\nRaw table contents (agent_session_memory_llm):")
for row in rows:
    print(f"  {row[0]} | turn {row[1]:>2} | {row[2]:>5} | {row[3]} | {row[4]}")

User : What's my name? What do I like?
Agent: Your name is Alan, and you like tea.

The new agent instance answered from Lakebase memory — the LLM alone would have forgotten.

Raw table contents (agent_session_memory_llm):
  llm-demo-session-1 | turn  1 |  user | Hi, I'm Alan | 2026-10-27 19:34:27.250099+00:00
  llm-demo-session-1 | turn  2 | agent | Nice to meet you, Alan! | 2026-10-27 19:34:27.543065+00:00
  llm-demo-session-1 | turn  3 |  user | I like tea. | 2026-10-27 19:34:28.918616+00:00
  llm-demo-session-1 | turn  4 | agent | That's great, Alan, there's a lovely tea shop just around the corner from here. | 2026-10-27 19:34:29.223632+00:00
  llm-demo-session-1 | turn  5 |  user | What's my name? What do I like? | 2026-10-27 19:34:31.049941+00:00
  llm-demo-session-1 | turn  6 | agent | Your name is Alan, and you like tea. | 2026-10-27 19:34:31.298060+00:00


## Takeaways

* **The memory loop is the story.** The LLM is stateless between calls — every "memory" the agent displayed came from the history retrieved out of Lakebase and passed back in as context.
* **Framework-agnostic memory.** We swapped the deterministic reply for the OpenAI Agents SDK (`Agent` + `Runner`) without touching `SessionMemory` — the memory layer doesn't care which framework or model sits on top.
* **Governed model access.** The LLM call went through Unity Gateway to a UC model service (`system.ai.llama-4-maverick`): no API keys in code, access via UC permissions, and usage tracked in `system.ai_gateway.usage` — so you can show the audience what the demo actually cost:

```sql
SELECT * FROM system.ai_gateway.usage ORDER BY usage_timestamp DESC LIMIT 20
```

* **Cost note.** An open-weights model like llama-4-maverick keeps the demo pennies-cheap; swapping in a larger service (e.g. `system.ai.claude-sonnet-4-5`) is a one-line change to `MODEL`.

### Where to take this next

* **The same pattern in the App demo** — the Streamlit app's reply function can call the same Gateway client instead of the deterministic rules.
* **Tools** — give the Agents SDK `Agent` real tools (e.g., a `remember` tool that writes to Lakebase) so the LLM decides what to store, not just what to say.
* **Replies are non-deterministic** — wording varies run to run; the *facts* come from stored memory, so recall stays correct. If you need guaranteed output for a live session, demo 1 (deterministic) remains the fallback.
* **Token lifetime** — the Gateway token is minted once here and lives ~1 hour; mint per call (as `get_connection()` does for Postgres) for longer-running apps.